1. What is Generative AI and what are its primary use cases across industries?

Generative AI is a type of Artificial Intelligence that can create new content such as text, images, videos, music, or code based on patterns it has learned from existing data. Instead of only analyzing data, it generates new outputs that resemble human-created content.
It is usually built using advanced models like Generative Adversarial Networks, Variational Autoencoders, and Transformer Models.

Primary Use Cases of Generative AI Across Industries
1. Healthcare - AI models help researchers simulate new medicines faster.
2. Marketing & Advertising  - Companies use AI to produce large volumes of marketing content quickly.
3. Software Development - Tools like GitHub Copilot help developers write code faster.
4. Media & Entertainment - Generative AI can create visual effects and digital characters.
5. Finance & Banking - Banks use AI to generate reports and analyze financial data quickly.
6. Education - AI helps create customized learning experiences for students.
And many more Industries.

2. Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

Probabilistic modeling is the foundation of generative models. These models try to learn the probability distribution of the data so they can generate new samples similar to the original data.
P(X,Y) where X = input data and Y = labels or outputs
By learning this distribution, the model can:
Generate new data samples similar to the training data.Model uncertainty in predictions.Understand how the data is generated.

Generative Model - IT is  Probability learned Learn P(X, Y) (joint distribution). It Goals is Model how data is generated. it Capable of to generate new data samples. ex- GAN, VAE, Naive Bayes
Discriminative Model - It Learn P(Y | X) (conditional probability). It is use to Directly classify or predict labels. it Cannot generate data. ex- Logistic Regression, SVM, Neural Networks

3. What is the difference between Autoencoders and Variational
Autoencoders (VAEs) in the context of text generation?

An Autoencoder is a neural network used to compress input data into a smaller representation (latent vector) and then reconstruct the original data.
Structure of an Autoencoder
Encoder , Latent Space , Decoder
In text generation, autoencoders can: Learn representations of sentences or documents.Reconstruct input text from encoded vectors.
Help in tasks like text summarization, denoising, and feature extraction.
Variational Autoencoders (VAE) -  A Variational Autoencoder (VAE) is an extension of the autoencoder that uses probabilistic modeling to generate new data.Instead of mapping input to a fixed latent vector, a VAE learns a probability distribution of the latent space (usually Gaussian).
In text generation, VAEs can:Generate new sentences similar to training data.Produce diverse text outputs. Model uncertainty and variability in language

4. Describe the working of attention mechanisms in Neural Machine
Translation (NMT). Why are they critical?

In Neural Machine Translation (NMT), the attention mechanism allows the model to focus on the most relevant words in the input sentence while generating each word in the translated sentence.

Traditional NMT systems used an encoder–decoder architecture where the encoder compresses the entire input sentence into a single vector. This often caused information loss, especially for long sentences.

The attention mechanism solves this problem by allowing the decoder to look at different parts of the input sentence dynamically during translation.
Attention Works in NMT
1. Encoding the Input Sentence
2. Calculating Attention Scores
3. Creating Attention Weights
4. Context Vector Creation
5. Generating the Output Word

5. What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

When using Generative AI for creative content like poetry or storytelling, several ethical considerations must be addressed to ensure responsible and fair use.
1. Copyright and Intellectual Property
Generative AI models are often trained on large datasets that may include copyrighted works. This raises questions about:
-Who owns the generated content?
-Whether AI outputs resemble existing works too closely.
-Fair compensation for original creators.
2. Plagiarism and Originality
AI-generated creative content should not replicate existing works without attribution. Ethical use requires:
-Ensuring originality of generated content
-Avoiding direct copying of existing literary works
-Transparency about AI assistance
3. Bias and Fair Representation
Generative AI systems learn from historical data, which may contain social or cultural biases. As a result:
-Stories may reinforce stereotypes.
-Certain groups may be unfairly represented.
Developers must monitor and reduce these biases to ensure fair and inclusive storytelling.
4. Transparency and Disclosure
Readers should know whether content was written by humans or generated by AI. Ethical practice includes:
-Disclosing the use of AI tools in content creation
-Avoiding misleading audiences about authorship

When using generative AI for creative content, key ethical considerations include copyright protection, originality, bias reduction, transparency, prevention of misuse, and respect for human creators. Responsible use ensures that AI enhances creativity without harming creators or audiences.


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.layers import Dense, Lambda
from tensorflow.keras.models import Model as KerasModel # Renamed Model to avoid conflict
from tensorflow.keras.losses import mse

# Dataset
sentences = [
"The sky is blue",
"The sun is bright",
"The grass is green",
"The night is dark",
"The stars are shining"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)
max_len = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')
vocab_size = len(tokenizer.word_index) + 1

print("Word Index:", tokenizer.word_index)
print("Padded Sequences:\n", padded_sequences)

input_dim = max_len
latent_dim = 2

# Define the VAE as a Keras Model subclass
class VAE(tf.keras.Model):
    def __init__(self, original_dim, latent_dim=2, intermediate_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.encoder_h = Dense(intermediate_dim, activation='relu')
        self.z_mean_layer = Dense(latent_dim)
        self.z_log_var_layer = Dense(latent_dim)

        # Decoder layers
        self.decoder_h = Dense(intermediate_dim, activation='relu')
        self.decoder_output = Dense(original_dim, activation='linear')

        self.latent_dim = latent_dim # Store latent_dim for sampling

    def call(self, inputs):
        # Encoder
        h = self.encoder_h(inputs)
        z_mean = self.z_mean_layer(h)
        z_log_var = self.z_log_var_layer(h)

        # Sampling from latent space
        epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], self.latent_dim))
        z = z_mean + tf.keras.ops.exp(0.5 * z_log_var) * epsilon

        # Decoder
        h_decoded = self.decoder_h(z)
        outputs = self.decoder_output(h_decoded)

        # Calculate VAE loss components
        reconstruction_loss = mse(inputs, outputs) # This is a KerasTensor of shape (batch_size,)
        kl_loss = -0.5 * tf.keras.ops.mean(
            1 + z_log_var - tf.keras.ops.square(z_mean) - tf.keras.ops.exp(z_log_var), axis=-1
        ) # This is a KerasTensor of shape (batch_size,)

        # Add the combined loss to the model. Keras will aggregate these per-sample losses.
        self.add_loss(reconstruction_loss + kl_loss)

        return outputs # The model's primary output is the reconstruction

# Instantiate and compile the VAE model
vae = VAE(original_dim=input_dim, latent_dim=latent_dim)
vae.compile(optimizer='adam') # No loss specified here, as it's handled via self.add_loss in call

# Build the model explicitly before calling summary()
vae.build(input_shape=(None, input_dim))
vae.summary()

vae.fit(padded_sequences,
        padded_sequences, # target is the input itself for autoencoder
        epochs=200,
        batch_size=2)

reconstructed = vae.predict(padded_sequences)

print("Reconstructed vectors:\n", reconstructed)

Word Index: {'the': 1, 'is': 2, 'sky': 3, 'blue': 4, 'sun': 5, 'bright': 6, 'grass': 7, 'green': 8, 'night': 9, 'dark': 10, 'stars': 11, 'are': 12, 'shining': 13}
Padded Sequences:
 [[ 1  3  2  4]
 [ 1  5  2  6]
 [ 1  7  2  8]
 [ 1  9  2 10]
 [ 1 11 12 13]]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'vae', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "vae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 81.2265
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 80.5158 
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 76.6331 
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 98.4997
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 92.5205
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 72.9324 
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 88.3849  
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 87.0148  
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 67.7723 
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 88.6016
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 78.5904  
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 78.6412  
Epoch 13/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 76.5348
Epoch 14/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 62.3326
Epoch 15/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 77.8141 

In [ ]:
from transformers import pipeline

# Load text generation pipeline
translator = pipeline("text-generation", model="gpt2")

text = "Translate the following English text to French and German: \
Artificial Intelligence is transforming many industries."

result = translator(text, max_length=100)

print(result[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate the following English text to French and German: Artificial Intelligence is transforming many industries. It is creating new jobs, but the jobs are now much more automated. I am now using artificial intelligence to create new jobs in the health care sector. I hope that many more companies will use it to create new jobs."

The report also says that companies in the health care space are starting to recognize that AI can be used to improve their performance, especially in the healthcare sector.

"In the healthcare sector, it is becoming increasingly clear that the health care sector is an important and vibrant sector of the economy, that it is growing at an accelerating rate, and that AI can be used to improve performance on a regular basis," it says.
